# Install

In [ ]:
!git clone -b dev https://github.com/lqb464/Sketch-to-Image-by-Pix2Pix

import os
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

os.chdir('Sketch-to-Image-by-Pix2Pix/')
!pip install -q -r requirements.txt

# Training

-   `python train.py --dataroot ./datasets/edges2shoes --name shoes_unet256_lambda100_0610 --model pix2pix --direction AtoB`

Change the `--dataroot` and `--name` to your own dataset's path and model's name. Use `--gpu_ids 0,1,..` to train on multiple GPUs and `--batch_size` to change the batch size. Add `--direction AtoB` if you want to train a model to transfrom from class B (Image) to A (Sketch).

In [ ]:
# tải một dataset nào đó từ repo gốc
# danh sách gồm: cityscapes, night2day, edges2handbags, edges2shoes, facades, maps

# !bash ./datasets/download_pix2pix_dataset.sh edges2shoes > download.log 2>&1

In [ ]:
# Tải model pretrain từ repo gốc
# Hoặc tự thêm checkpoint của mình vào ./checkpoints/{NAME}_pretrained/latest_net_G.pt
# Danh sách có sẵn gồm: edges2shoes, sat2map, map2sat, facades_label2photo, and day2night

# !bash ./scripts/download_pix2pix_model.sh facades_label2photo > download.log 2>&1

In [ ]:
import yaml, random, numpy as np, torch

# Chọn dataset muốn train
CONFIG_PATH = "./configs/baseline_facades.yaml"
# CONFIG_PATH = "./configs/baseline_edges2shoes.yaml"
# CONFIG_PATH = "./configs/baseline_cufs.yaml"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Fix seed từ config
SEED = cfg.get("seed", 42)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Loaded config: {cfg['name']} | Seed: {SEED}")

In [ ]:
!python train.py \
  --dataroot {cfg['dataroot']} \
  --name {cfg['name']} \
  --model {cfg['model']} \
  --direction {cfg['direction']} \
  --netG {cfg['netG']} \
  --netD {cfg['netD']} \
  --lambda_L1 {cfg['lambda_L1']} \
  --batch_size {cfg['batch_size']} \
  --lr {cfg['lr']} \
  --n_epochs {cfg['n_epochs']} \
  --n_epochs_decay {cfg['n_epochs_decay']} \
  --num_threads {cfg['num_threads']} \
  --load_size {cfg['load_size']} \
  --crop_size {cfg['crop_size']} \
  --print_freq {cfg['print_freq']} \
  --display_freq {cfg['display_freq']} \
  --save_latest_freq {cfg['save_latest_freq']} \
  --save_epoch_freq {cfg['save_epoch_freq']} \
  --use_wandb \
  --no_html

# Testing

-   `python test.py --dataroot ./datasets/facades --direction BtoA --model pix2pix --name facades_unet256_lambda100_0610`

Change the `--dataroot`, `--name`, and `--direction` to be consistent with your trained model's configuration and how you want to transform images.

> Note that we specified --direction BtoA as Facades dataset's A to B direction is photos to labels.
> If you would like to apply a pre-trained model to a collection of input images (rather than image pairs), please use --model test option. See ./scripts/test_single.sh for how to apply a model to Facade label maps (stored in the directory facades/testB).

In [ ]:
!python test.py \
  --dataroot {cfg['dataroot']} \
  --name {cfg['name']} \
  --model {cfg['model']} \
  --direction {cfg['direction']} \
  --load_size {cfg['load_size']} \
  --crop_size {cfg['crop_size']} \
  --num_test {cfg['num_test']} \
  --use_wandb

# Visualize

In [ ]:
import matplotlib.pyplot as plt

base = f"./results/{cfg['name']}/test_latest/images/"
sample = '3'

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
titles = ['Input (Sketch)', 'Generated (Fake)', 'Ground Truth (Real)']
files  = [f'{sample}_real_A.png', f'{sample}_fake_B.png', f'{sample}_real_B.png']

for ax, title, fname in zip(axes, titles, files):
    ax.imshow(plt.imread(base + fname))
    ax.set_title(title)
    ax.axis('off')

plt.tight_layout()
plt.show()